## Cell 3 — Serve via Cloudflare Tunnel

No signup, no account, no password needed.

Gives you a public URL like `https://abc-xyz-123.trycloudflare.com`

Open it in any browser — the full UI works. The URL changes each session, that is normal.

## Cell 1 — Setup + verify GPU

In [ ]:
!git clone https://github.com/Kaur-Simarpreet/molecular-design-vae.git
%cd molecular-design-vae
!pip install -q torch selfies flask flask-cors scipy requests
!pip install -q rdkit

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU not detected. Runtime > Change runtime type > T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete')

## Cell 1b — Set up real DiffDock docking (~10 min, ~3 GB)

Optional but recommended for Large GPU mode. Downloads DiffDock + ESM-2 weights for state-of-the-art blind docking. Without this, scoring uses mock estimates.

In [ ]:
!bash setup_docking.sh --diffdock
!python docking.py   # verify


## Cell 2 — Train (20-40 min on T4 GPU)

First run downloads ZINC-250K (~8 min, then cached).

In [ ]:
!python train_vae_extended.py --mode large-gpu

## Cell 3 — Serve via Cloudflare Tunnel

No signup, no account, no password needed.

Gives you a public URL like `https://abc-xyz-123.trycloudflare.com`

Open it in any browser — the full UI works. The URL changes each session, that is normal.

In [ ]:
import subprocess, threading, time, re

# Start serve.py in background
def run_server():
    subprocess.run(['python', 'serve.py', '--host', '0.0.0.0', '--port', '5000'])

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(10)

# Download cloudflared — no account, no signup needed
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', 'cloudflared'
], check=True)
subprocess.run(['chmod', '+x', 'cloudflared'])

# Start Cloudflare tunnel
proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stderr=subprocess.PIPE, text=True
)

# Wait for public URL
print('Starting tunnel — please wait ~15 seconds...')
for line in proc.stderr:
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        print(f'\n=== UI available at: {match.group(0)} ===')
        print('Open this URL in any browser.')
        print('No password. No signup. All 8 tabs work.')
        break


## Cell 4 — Download trained model (4.9M params, ~20MB)

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('saved_model', 'zip', 'saved_model')
files.download('saved_model.zip')